In [2]:
import requests
import pandas as pd
import time
from tqdm import tqdm

In [3]:
subjects = [
    "fantasy",
    "science_fiction",
    "young_adult",
    "historical_fiction",
    "mystery",
    "thriller",
    "horror"
]

In [4]:
def get_subject_books(subject, limit=1000):

    url = f"https://openlibrary.org/subjects/{subject}.json"

    response = requests.get(
        url,
        params={"limit": limit},
        timeout=30
    )

    data = response.json()

    books = []

    for work in data.get("works", []):

        books.append({
            "title": work.get("title", ""),
            "author": (
                work["authors"][0]["name"]
                if work.get("authors")
                else ""
            ),
            "work_key": work.get("key", "")
        })

    return books

In [5]:
def get_subject_books(subject, limit=1000):

    url = f"https://openlibrary.org/subjects/{subject}.json"

    response = requests.get(
        url,
        params={"limit": limit},
        timeout=30
    )

    data = response.json()

    books = []

    for work in data.get("works", []):

        books.append({
            "title": work.get("title"),
            "author": (
                work["authors"][0]["name"]
                if work.get("authors")
                else ""
            ),
            "work_key": work.get("key")
        })

    return books

In [6]:
books = get_subject_books(
    "fantasy",
    limit=5
)

print(books)

[{'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'work_key': '/works/OL138052W'}, {'title': 'The Wonderful Wizard of Oz', 'author': 'L. Frank Baum', 'work_key': '/works/OL18417W'}, {'title': 'Treasure Island', 'author': 'Robert Louis Stevenson', 'work_key': '/works/OL24034W'}, {'title': "Gulliver's Travels", 'author': 'Jonathan Swift', 'work_key': '/works/OL20600W'}, {'title': "A Midsummer Night's Dream", 'author': 'William Shakespeare', 'work_key': '/works/OL259010W'}]


In [8]:
catalog = pd.DataFrame(all_books)

print(catalog.shape)

catalog.head()

NameError: name 'all_books' is not defined

In [ ]:
catalog = catalog.drop_duplicates(
    subset=["work_key"]
)

print(catalog.shape)

(1000, 3)


In [ ]:
import os
os.makedirs(
    "../data/catalog",
    exist_ok=True
)

catalog.to_csv(
    "../data/catalog/catalog_v1.csv",
    index=False
)

print("Saved!")

Saved!


In [ ]:
def get_metadata(work_key):

    try:

        url = (
            f"https://openlibrary.org"
            f"{work_key}.json"
        )

        data = requests.get(
            url,
            timeout=20
        ).json()

        description = ""

        if isinstance(
            data.get("description"),
            dict
        ):
            description = (
                data["description"]
                .get("value", "")
            )

        elif isinstance(
            data.get("description"),
            str
        ):
            description = data["description"]

        subjects = data.get(
            "subjects",
            []
        )

        return description, subjects

    except:

        return "", []

In [ ]:
row = catalog.iloc[0]

description, subjects = get_metadata(
    row["work_key"]
)

print(row["title"])
print()
print(description[:500])
print()
print(subjects[:10])

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

In [ ]:
def fetch_metadata(work_key):

    desc, subs = get_metadata(work_key)

    return {
        "work_key": work_key,
        "description": desc,
        "genres": "|".join(subs)
    }

In [ ]:
results = []

with ThreadPoolExecutor(max_workers=10) as executor:

    futures = [
        executor.submit(
            fetch_metadata,
            wk
        )
        for wk in catalog["work_key"]
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        results.append(
            future.result()
        )

100%|██████████| 6120/6120 [54:53<00:00,  1.86it/s]  


In [ ]:
meta_df = pd.DataFrame(results)

meta_df.head()

,work_key,description,genres
0,/works/OL1089297W,The Prince (Italian: Il Principe [il ˈprintʃip...,"Political science, early works to 1800|Machiav..."
1,/works/OL18417W,"Over a century after its initial publication, ...",Witches|Spanish language materials|Fiction|Wiz...
2,/works/OL20600W,A parody of traveler’s tales and a satire of h...,YA|Young adult|Juvenile|Fiction|Fantasy|Utopia...
3,/works/OL18396W,"Tip and his creation, Jack Pumpkin, run away t...","Children's stories, American|Juvenile fiction|..."
4,/works/OL10263W,*Le Petit Prince* est une œuvre de langue fran...,adventure|fantasy|friendship|love|childhood|lo...


In [ ]:
catalog = catalog.merge(
    meta_df,
    on="work_key",
    how="left"
)

catalog.head()

,title,author,work_key,description,genres
0,Alice's Adventures in Wonderland,Lewis Carroll,/works/OL138052W,,
1,The Wonderful Wizard of Oz,L. Frank Baum,/works/OL18417W,"Over a century after its initial publication, ...",Witches|Spanish language materials|Fiction|Wiz...
2,Treasure Island,Robert Louis Stevenson,/works/OL24034W,Traditionally considered a coming-of-age story...,Fiction|Treasure Island (Imaginary place)|Trea...
3,Gulliver's Travels,Jonathan Swift,/works/OL20600W,A parody of traveler’s tales and a satire of h...,YA|Young adult|Juvenile|Fiction|Fantasy|Utopia...
4,A Midsummer Night's Dream,William Shakespeare,/works/OL259010W,,


In [ ]:
catalog.to_csv(
    "../data/catalog/catalog_full.csv",
    index=False
)

In [ ]:
print("Books:", len(catalog))

print(
    "Descriptions:",
    catalog["description"]
    .notna()
    .sum()
)

print(
    "Genres:",
    catalog["genres"]
    .notna()
    .sum()
)

Books: 6120
Descriptions: 6120
Genres: 6120


In [ ]:
print("Books:", len(catalog))

print(
    "Descriptions:",
    (catalog["description"].fillna("") != "").sum()
)

print(
    "Genres:",
    (catalog["genres"].fillna("") != "").sum()
)

Books: 6120
Descriptions: 5324
Genres: 5826


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\anshu\Documents\codes\ML\BookRecomendation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\anshu\Documents\codes\ML\BookRecomendation\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anshu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or

In [ ]:
catalog["combined_text"] = (
    catalog["title"].fillna("") + " " +
    catalog["author"].fillna("") + " " +
    catalog["description"].fillna("") + " " +
    catalog["genres"].fillna("")
)

KeyError: 'description'

In [ ]:
embeddings = model.encode(
    catalog["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|██████████| 96/96 [01:27<00:00,  1.10it/s]


In [ ]:
print(embeddings.shape)

(6120, 384)


In [ ]:
import numpy as np

np.save(
    "../models/book_embeddings.npy",
    embeddings
)

catalog.to_csv(
    "../models/book_catalog.csv",
    index=False
)

In [ ]:
from rapidfuzz import process, fuzz

titles = catalog["title"].fillna("").tolist()

def match_book(query):

    match = process.extractOne(
        query,
        titles,
        scorer=fuzz.token_sort_ratio
    )

    return match

In [ ]:
print(match_book("Cytonic"))
print(match_book("Oathbringer"))
print(match_book("Starsight"))
print(match_book("Hero of Ages"))

('Cryptonomicon', 70.0, 1243)
('Oathbringer', 100.0, 810)
('Starlight', 88.88888888888889, 593)
('The Hero of Ages', 85.71428571428572, 430)


In [ ]:
catalog[
    catalog["title"]
    .str.contains(
        "Cytonic",
        case=False,
        na=False
    )
]

,title,author,work_key,description,genres,combined_text


In [ ]:
catalog[
    catalog["title"]
    .str.contains(
        "Starsight",
        case=False,
        na=False
    )
]

,title,author,work_key,description,genres,combined_text


In [ ]:
import pandas as pd

df = pd.read_csv(
    "../outputs/detected_books.csv"
)

print(df.shape)
df.head()

(165, 6)


,file,ocr,title,author,corrected_spine_text,confidence
0,..\outputs\book_spines\spine_101.jpg,SANBRSONCYTONI,Cytonic,Brandon Sanderson,BRANDON SANDERSON CYTONIC,1.0
1,..\outputs\book_spines\spine_10.jpg,NTHON RYA LEGION OPFLAM,Legion of Flame,Anthony Ryan,ANTHONY RYAN LEGION OF FLAME,1.0
2,..\outputs\book_spines\spine_100.jpg,KEINSRZLNLA,Wintersmith,Terry Pratchett,WINTERSMITH Terry Pratchett,1.0
3,..\outputs\book_spines\spine_1.jpg,IXHERO SANDERSON ANDOI E OF AGES,The Hero of Ages,Brandon Sanderson,The Hero of Ages BRANDON SANDERSON TOR,1.0
4,..\outputs\book_spines\spine_102.jpg,HE MALEFICENT SEVEN,The Maleficent Seven,चरणन जोन्स्टन (Charanan Johnston / Likely an e...,THE MALEFICENT SEVEN,1.0


In [ ]:
detected_books = (
    df["title"]
    .dropna()
    .astype(str)
    .tolist()
)

print(len(detected_books))

165


In [ ]:
from rapidfuzz import process, fuzz

titles = catalog["title"].fillna("").tolist()

matches = 0

for title in detected_books:

    result = process.extractOne(
        title,
        titles,
        scorer=fuzz.token_sort_ratio
    )

    if result[1] >= 90:
        matches += 1

print("Matches:", matches)
print("Total:", len(detected_books))
print("Coverage:", matches / len(detected_books))

Matches: 55
Total: 165
Coverage: 0.3333333333333333


In [ ]:
missing = []

for title in detected_books:

    result = process.extractOne(
        title,
        titles,
        scorer=fuzz.token_sort_ratio
    )

    if result[1] < 90:

        missing.append(
            (title, result[0], result[1])
        )

missing[:30]

NameError: name 'detected_books' is not defined

In [ ]:
fantasy_authors = [
    "Brandon Sanderson",
    "Robert Jordan",
    "Robin Hobb",
    "Joe Abercrombie",
    "John Gwynne",
    "Anthony Ryan",
    "Steven Erikson",
    "Patrick Rothfuss",
    "Mark Lawrence",
    "Brent Weeks",
    "Brian McClellan",
    "Andrzej Sapkowski",
    "Terry Brooks",
    "Raymond E. Feist",
    "David Gemmell",
    "Guy Gavriel Kay",
    "Tad Williams",
    "Michael J. Sullivan",
    "Jay Kristoff",
    "Brian Staveley",
    "R. F. Kuang",
    "Evan Winter",
    "James Islington",
    "Pierce Brown",
    "Christopher Ruocchio"
]

In [ ]:
scifi_authors = [
    "Peter F. Hamilton",
    "Alastair Reynolds",
    "Ann Leckie",
    "Adrian Tchaikovsky",
    "Isaac Asimov",
    "Arthur C. Clarke",
    "Frank Herbert",
    "Philip K. Dick",
    "Orson Scott Card",
    "John Scalzi",
    "Neal Stephenson",
    "Iain M. Banks",
    "Dan Simmons",
    "Kim Stanley Robinson",
    "Vernor Vinge",
    "Martha Wells",
    "Lois McMaster Bujold",
    "Becky Chambers",
    "Joe Haldeman",
    "Larry Niven"
]

In [ ]:
classic_authors = [
    "J. R. R. Tolkien",
    "C. S. Lewis",
    "Ursula K. Le Guin",
    "Roger Zelazny",
    "Michael Moorcock",
    "Gene Wolfe",
    "Terry Pratchett",
    "Neil Gaiman",
    "Stephen Donaldson",
    "Anne McCaffrey"
]

In [ ]:
popular_authors = [
    "Stephen King",
    "Dean Koontz",
    "Dan Brown",
    "Michael Crichton",
    "Ken Follett",
    "John Grisham",
    "Tom Clancy",
    "Lee Child",
    "James Patterson",
    "David Baldacci",
    "Harlan Coben",
    "George R. R. Martin",
    "Suzanne Collins",
    "J. K. Rowling",
    "Rick Riordan"
]

In [ ]:
ya_authors = [
    "Brandon Mull",
    "Cassandra Clare",
    "Leigh Bardugo",
    "Sarah J. Maas",
    "Holly Black",
    "Tahereh Mafi",
    "Veronica Roth",
    "Marie Lu",
    "Scott Westerfeld",
    "Christopher Paolini"
]

In [ ]:
authors1 = [

    # Fantasy
    "Brandon Sanderson",
    "Robert Jordan",
    "J.R.R. Tolkien",
    "George R.R. Martin",
    "Joe Abercrombie",
    "Robin Hobb",
    "Terry Pratchett",
    "Patrick Rothfuss",
    "Steven Erikson",
    "Raymond E. Feist",
    "David Gemmell",
    "John Gwynne",
    "Anthony Ryan",
    "Mark Lawrence",
    "Jay Kristoff",
    "Andrzej Sapkowski",
    "Guy Gavriel Kay",
    "Brent Weeks",
    "Brian McClellan",
    "James Islington",
    "Michael J. Sullivan",
    "Tad Williams",
    "R.A. Salvatore",
    "Ursula K. Le Guin",
    "Terry Brooks",

    # Science Fiction
    "Isaac Asimov",
    "Arthur C. Clarke",
    "Frank Herbert",
    "Philip K. Dick",
    "Alastair Reynolds",
    "Peter F. Hamilton",
    "Ann Leckie",
    "Adrian Tchaikovsky",
    "Iain M. Banks",
    "John Scalzi",
    "Neal Stephenson",
    "Dan Simmons",
    "Orson Scott Card",
    "Joe Haldeman",
    "Kim Stanley Robinson",
    "Cixin Liu",
    "Martha Wells",
    "Andy Weir",
    "Becky Chambers",
    "William Gibson",

    # Horror
    "Stephen King",
    "Clive Barker",
    "Dean Koontz",
    "H.P. Lovecraft",
    "Shirley Jackson",

    # Thriller / Mystery
    "Agatha Christie",
    "Arthur Conan Doyle",
    "Dan Brown",
    "Lee Child",
    "Michael Connelly",
    "John Grisham",
    "Stieg Larsson",
    "Gillian Flynn",
    "Tana French",
    "Harlan Coben",

    # Literary / Modern
    "Haruki Murakami",
    "Kazuo Ishiguro",
    "Margaret Atwood",
    "Donna Tartt",
    "Cormac McCarthy",
    "Kurt Vonnegut",
    "Neil Gaiman",
    "Toni Morrison",
    "George Orwell",
    "Aldous Huxley",

    # YA
    "J.K. Rowling",
    "Suzanne Collins",
    "Rick Riordan",
    "Cassandra Clare",
    "Leigh Bardugo",
    "Sarah J. Maas",
    "Veronica Roth",
    "Holly Black",
    "Christopher Paolini",
    "Philip Pullman",

    # Classics
    "Jane Austen",
    "Charles Dickens",
    "Mark Twain",
    "Leo Tolstoy",
    "Fyodor Dostoevsky",
    "Victor Hugo",
    "Jules Verne",
    "H.G. Wells",
    "Herman Melville",
    "Oscar Wilde",

    # Manga / Light Novels
    "Eiichiro Oda",
    "Masashi Kishimoto",
    "Tite Kubo",
    "Hajime Isayama",
    "Tsugumi Ohba",

    # Popular Contemporary
    "Colleen Hoover",
    "Taylor Jenkins Reid",
    "Emily Henry",
    "Rebecca Yarros",
    "Kristin Hannah",
    "Paulo Coelho",
    "Nicholas Sparks",
    "Ken Follett",
    "Diana Gabaldon",
    "James Patterson"
]

In [ ]:
more_authors = [

    # Fantasy
    "Naomi Novik",
    "N.K. Jemisin",
    "R.F. Kuang",
    "Samantha Shannon",
    "Brian Staveley",
    "Daniel Abraham",
    "Jenn Lyons",
    "Miles Cameron",
    "Scott Lynch",
    "Kevin Hearne",
    "Katherine Arden",
    "Gene Wolfe",
    "Glen Cook",
    "Roger Zelazny",
    "Anne McCaffrey",
    "Mercedes Lackey",
    "Robin McKinley",
    "T. Kingfisher",
    "Juliet Marillier",
    "Seanan McGuire",

    # Sci-Fi
    "Vernor Vinge",
    "Greg Egan",
    "Peter Watts",
    "Charles Stross",
    "Robert A. Heinlein",
    "Larry Niven",
    "Poul Anderson",
    "David Brin",
    "Lois McMaster Bujold",
    "C.J. Cherryh",
    "Ben Bova",
    "Jack McDevitt",
    "Frederik Pohl",
    "James S.A. Corey",
    "Richard K. Morgan",
    "Walter M. Miller Jr.",
    "Ted Chiang",
    "Robert Charles Wilson",
    "Nancy Kress",
    "Connie Willis",

    # Mystery / Crime
    "Jo Nesbo",
    "Dennis Lehane",
    "Patricia Cornwell",
    "Ruth Rendell",
    "P.D. James",
    "Louise Penny",
    "Robert Galbraith",
    "Ian Rankin",
    "Elly Griffiths",
    "Val McDermid",

    # Thriller
    "Tom Clancy",
    "Vince Flynn",
    "Brad Thor",
    "David Baldacci",
    "Robert Ludlum",
    "Frederick Forsyth",
    "Daniel Silva",
    "Clive Cussler",
    "Jack Carr",
    "Steve Berry",

    # Romance
    "Nora Roberts",
    "Julia Quinn",
    "Lisa Kleypas",
    "Tessa Dare",
    "Christina Lauren",
    "Ali Hazelwood",
    "Abby Jimenez",
    "Jennifer L. Armentrout",
    "Helen Hoang",
    "Colleen Hoover",

    # YA / Fantasy YA
    "Rainbow Rowell",
    "Marie Lu",
    "Tahereh Mafi",
    "Victoria Aveyard",
    "Sabaa Tahir",
    "Maggie Stiefvater",
    "Amie Kaufman",
    "Jay Kristoff",
    "Erin Morgenstern",
    "Laini Taylor",

    # Literary Fiction
    "Salman Rushdie",
    "John Steinbeck",
    "Ernest Hemingway",
    "William Faulkner",
    "Virginia Woolf",
    "James Joyce",
    "Gabriel Garcia Marquez",
    "Isabel Allende",
    "Jhumpa Lahiri",
    "Zadie Smith",

    # Historical Fiction
    "Hilary Mantel",
    "Bernard Cornwell",
    "Conn Iggulden",
    "Sharon Kay Penman",
    "Philippa Gregory",
    "Ken Follett",
    "Edward Rutherfurd",
    "Robert Harris",
    "Simon Scarrow",
    "Christian Cameron",

    # Horror
    "Joe Hill",
    "Paul Tremblay",
    "Adam Nevill",
    "Ramsey Campbell",
    "Richard Matheson",
    "Robert McCammon",
    "Nick Cutter",
    "Grady Hendrix",
    "Alma Katsu",
    "Josh Malerman",

    # Manga
    "Akira Toriyama",
    "Kentaro Miura",
    "Naoki Urasawa",
    "Takehiko Inoue",
    "Yoshihiro Togashi",
    "Gege Akutami",
    "Koyoharu Gotouge",
    "Tatsuki Fujimoto",
    "Yuki Tabata",
    "Makoto Yukimura",

    # Classics
    "Alexandre Dumas",
    "Edgar Allan Poe",
    "Bram Stoker",
    "Robert Louis Stevenson",
    "Mary Shelley",
    "Daniel Defoe",
    "Homer",
    "Dante Alighieri",
    "Miguel de Cervantes",
    "Geoffrey Chaucer",

    # Popular Modern
    "Brandon Mull",
    "Brandon Mull",
    "Brent Weeks",
    "Pierce Brown",
    "Will Wight",
    "T.J. Klune",
    "Rebecca Ross",
    "Fonda Lee",
    "John Bierce",
    "Ryan Cahill",

    # Huge sellers
    "John Green",
    "E.L. James",
    "Paula Hawkins",
    "Delia Owens",
    "Anthony Doerr",
    "Andy Weir",
    "Matt Haig",
    "Amor Towles",
    "Fredrik Backman",
    "Richard Osman",

    # Nonfiction giants
    "Yuval Noah Harari",
    "Malcolm Gladwell",
    "Nassim Nicholas Taleb",
    "Daniel Kahneman",
    "James Clear",
    "Stephen Hawking",
    "Walter Isaacson",
    "Simon Sinek",
    "Robert Greene",
    "Ryan Holiday"
]

In [ ]:
authors = (
    fantasy_authors +
    scifi_authors +
    classic_authors +
    popular_authors +
    ya_authors +
    authors1 +
    more_authors
)

authors = sorted(
    list(set(authors))
)

print(len(authors))

270


In [ ]:
import requests

def get_author_key(author_name):

    url = "https://openlibrary.org/search/authors.json"

    params = {
        "q": author_name
    }

    try:

        data = requests.get(
            url,
            params=params,
            timeout=30
        ).json()

        docs = data.get("docs", [])

        if len(docs) == 0:
            return None

        return docs[0]["key"]

    except Exception as e:

        print(author_name, e)
        return None

In [ ]:
def get_books_by_author(author):

    r = requests.get(
        "https://openlibrary.org/search.json",
        params={
            "author": author,
            "limit": 1000
        },
        timeout=30
    )

    docs = r.json()["docs"]

    books = []

    for d in docs:

        books.append({
            "title": d.get("title"),
            "author": author
        })

    return books

In [ ]:
key = get_author_key(
    "Brandon Sanderson"
)

print(key)

NameError: name 'get_author_key' is not defined

In [ ]:
works=get_author_works(key)

print(len(works))
print(works[:10])

NameError: name 'get_author_works' is not defined

In [ ]:
author_catalog = pd.DataFrame(all_books)

print(author_catalog.shape)

author_catalog.head()

(1000, 3)


,title,author,work_key
0,Alice's Adventures in Wonderland,Lewis Carroll,/works/OL138052W
1,The Wonderful Wizard of Oz,L. Frank Baum,/works/OL18417W
2,Treasure Island,Robert Louis Stevenson,/works/OL24034W
3,Gulliver's Travels,Jonathan Swift,/works/OL20600W
4,A Midsummer Night's Dream,William Shakespeare,/works/OL259010W


In [ ]:
import pandas as pd

catalog_books = pd.read_csv(
    "../data/catalog_books.csv"
)

print(catalog_books.shape)

(10217, 4)


In [ ]:
def clean_title(x):

    if pd.isna(x):
        return ""

    return (
        str(x)
        .lower()
        .strip()
    )

author_catalog["title_clean"] = (
    author_catalog["title"]
    .apply(clean_title)
)

catalog_books["title_clean"] = (
    catalog_books["title"]
    .apply(clean_title)
)

In [ ]:
author_catalog = (
    author_catalog
    .drop_duplicates(
        subset=["title_clean"]
    )
)

print(author_catalog.shape)

(986, 4)


In [ ]:
new_books = author_catalog[
    ~author_catalog["title_clean"].isin(
        catalog_books["title_clean"]
    )
].copy()

print("New books:", len(new_books))

New books: 0


In [ ]:
master_catalog = pd.concat(
    [
        catalog_books,
        new_books
    ],
    ignore_index=True
)

In [ ]:
master_catalog = (
    master_catalog
    .drop_duplicates(
        subset=["title_clean"]
    )
    .reset_index(drop=True)
)

In [ ]:
print(
    "Original:",
    len(catalog_books)
)

print(
    "Added:",
    len(new_books)
)

print(
    "Final:",
    len(master_catalog)
)

Original: 10217
Added: 0
Final: 10217


In [ ]:
master_catalog.to_csv(
    "../data/processed/master_catalog.csv",
    index=False
)

In [ ]:
from rapidfuzz import process, fuzz

titles = (
    master_catalog["title"]
    .dropna()
    .tolist()
)

matches = 0

for book in detected_books:

    match = process.extractOne(
        book,
        titles,
        scorer=fuzz.token_sort_ratio
    )

    if match and match[1] >= 85:
        matches += 1

print(
    "Coverage:",
    matches / len(detected_books)
)

NameError: name 'detected_books' is not defined

In [ ]:
matched_books = []
unmatched_books = []

for title in detected_books:

    match = process.extractOne(
        title,
        master_catalog["title"].tolist(),
        scorer=fuzz.token_sort_ratio
    )

    if match and match[1] >= 85:
        matched_books.append(title)
    else:
        unmatched_books.append(title)

print("Matched:", len(matched_books))
print("Unmatched:", len(unmatched_books))

NameError: name 'detected_books' is not defined

In [ ]:
for book in unmatched_books[:50]:
    print(book)

In [ ]:
master_catalog[
    master_catalog["title"]
    .str.contains(
        "Rhythm of War",
        case=False,
        na=False
    )
][["title","author"]]

,title,author


In [ ]:
master_catalog[
    master_catalog["title"]
    .str.contains(
        "Rhythm of War",
        case=False,
        na=False
    )
][["title","author"]]

,title,author


In [ ]:
master_catalog[
    master_catalog["title"]
    .str.contains(
        "Cytonic",
        case=False,
        na=False
    )
][["title","author"]]

,title,author


In [ ]:
check_titles = [
    "Cytonic",
    "Rhythm of War",
    "Tress of the Emerald Sea",
    "The Lost Metal",
    "Mother of Learning",
    "The Last Wish",
    "Blood of Elves",
    "The Shadow of What Was Lost",
    "Dark Dawn",
    "Defiant"
]

for title in check_titles:

    matches = master_catalog[
        master_catalog["title"]
        .str.contains(
            title,
            case=False,
            na=False
        )
    ]

    print("\n", title)
    print(len(matches))

    if len(matches):
        print(
            matches[
                ["title","author"]
            ].head()
        )


 Cytonic
0

 Rhythm of War
0

 Tress of the Emerald Sea
0

 The Lost Metal
0

 Mother of Learning
0

 The Last Wish
0

 Blood of Elves
0

 The Shadow of What Was Lost
0

 Dark Dawn
0

 Defiant
1
                   title        author
1168  The Defiant Agents  Andre Norton


In [ ]:
import requests
import pandas as pd
import time

In [ ]:
def get_wiki_page(author):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": author + " bibliography",
        "format": "json"
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=30,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        if response.status_code != 200:
            return None

        data = response.json()

        results = data["query"]["search"]

        if len(results) == 0:
            return None

        return results[0]["title"]

    except Exception as e:

        print(author, e)
        return None

In [ ]:
def get_books_from_wikipedia(author):

    page = get_wiki_page(author)

    if page is None:
        return []

    try:

        url = (
            "https://en.wikipedia.org/wiki/"
            + page.replace(" ", "_")
        )

        tables = pd.read_html(url)

        books = []

        for table in tables:

            for col in table.columns:

                if "title" in str(col).lower():

                    titles = (
                        table[col]
                        .dropna()
                        .astype(str)
                        .tolist()
                    )

                    books.extend(titles)

        return books

    except Exception as e:

        print(author, e)
        return []

In [ ]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/Brandon_Sanderson_bibliography"

response = requests.get(
    url,
    headers={
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
)

print(response.status_code)

200


In [ ]:
import requests
import pandas as pd
from io import StringIO

url = "https://en.wikipedia.org/wiki/Brandon_Sanderson_bibliography"

response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

html = response.text

tables = pd.read_html(
    StringIO(html)
)

print("Tables:", len(tables))

Tables: 17


In [ ]:
for i, table in enumerate(tables):

    print("\n====================")
    print("TABLE", i)
    print("====================")

    print(table.head())


TABLE 0
                    Title  Year    First edition publisher             Notes  \
0                Elantris  2005                  Tor Books               NaN   
1              Warbreaker  2009                  Tor Books               NaN   
2          The Sunlit Man  2023  Dragonsteel Entertainment  Secret Project 4   
3  Isles of the Emberdark  2025  Dragonsteel Entertainment  Secret Project 5   

     Ref.  
0     [2]  
1  [2][3]  
2     [4]  
3     [5]  

TABLE 1
                   Title        Year First edition publisher        Ref.  \
               First Era   First Era               First Era   First Era   
0       The Final Empire        2006               Tor Books         [2]   
1  The Well of Ascension        2007               Tor Books         [2]   
2       The Hero of Ages        2008               Tor Books         [2]   
3             Second Era  Second Era              Second Era  Second Era   
4       The Alloy of Law        2011               Tor Books     

In [ ]:
from io import StringIO
import requests
import pandas as pd

def get_wiki_books(page):

    url = f"https://en.wikipedia.org/wiki/{page}"

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=30
    )

    tables = pd.read_html(
        StringIO(response.text)
    )

    books = []

    for table in tables:

        cols = [
            str(c).lower()
            for c in table.columns
        ]

        has_title = any(
            "title" in c
            for c in cols
        )

        if not has_title:
            continue

        title_col = None

        for col in table.columns:

            if "title" in str(col).lower():

                title_col = col
                break

        if title_col is None:
            continue

        titles = (
            table[title_col]
            .dropna()
            .astype(str)
            .tolist()
        )

        books.extend(titles)

    return books

In [ ]:
books = get_wiki_books(
    "Brandon_Sanderson_bibliography"
)

print(len(books))

for b in books[:50]:
    print(b)

86
Elantris
Warbreaker
The Sunlit Man
Isles of the Emberdark
The Final Empire
The Well of Ascension
The Hero of Ages
Second Era
The Alloy of Law
Shadows of Self
The Bands of Mourning
The Lost Metal
The Way of Kings
Words of Radiance
Oathbringer
Rhythm of War
Wind and Truth
Tress of the Emerald Sea
Yumi and the Nightmare Painter
The Fires of December
The Hope of Elantris
"The Eleventh Metal"
The Emperor's Soul
Shadows for Silence in the Forests of Hell
Sixth of the Dusk
"Allomancer Jak and the Pits of Eltania"
Secret History
Arcanum Unbounded: The Cosmere Collection
Edgedancer
Dawnshard
Elsecaller
King Lopen the First of Alethkar
White Sand I
White Sand II
White Sand III
White Sand Omnibus
Alcatraz Versus the Evil Librarians
Alcatraz Versus the Scrivener's Bones
Alcatraz Versus the Knights of Crystallia
Alcatraz Versus the Shattered Lens
Alcatraz Versus the Dark Talent
Bastille Versus the Evil Librarians
"Defending Elysium"
Skyward
Starsight
Sunreach
ReDawn
Cytonic
Evershore
Skyward Fli

In [ ]:
wiki_df = pd.DataFrame({
    "title": books,
    "author": "Brandon Sanderson"
})

wiki_df.head()

,title,author
0,Elantris,Brandon Sanderson
1,Warbreaker,Brandon Sanderson
2,The Sunlit Man,Brandon Sanderson
3,Isles of the Emberdark,Brandon Sanderson
4,The Final Empire,Brandon Sanderson


In [ ]:
import requests

def get_author_page(author):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": author,
        "format": "json"
    }

    try:

        response = requests.get(
            url,
            params=params,
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            timeout=30
        )

        data = response.json()

        results = data["query"]["search"]

        if len(results) == 0:
            return None

        return results[0]["title"]

    except:
        return None

In [ ]:
from io import StringIO
import pandas as pd

def get_books_from_page(page):

    url = (
        "https://en.wikipedia.org/wiki/"
        + page.replace(" ", "_")
    )

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=30
    )

    tables = pd.read_html(
        StringIO(response.text)
    )

    books = []

    for table in tables:

        for col in table.columns:

            if "title" in str(col).lower():

                books.extend(
                    table[col]
                    .dropna()
                    .astype(str)
                    .tolist()
                )

    return books

In [ ]:
page = get_author_page(
    "Brandon Sanderson"
)

print(page)

Brandon Sanderson


In [ ]:
import requests

url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "query",
    "list": "search",
    "srsearch": "Brandon Sanderson bibliography",
    "format": "json"
}

data = requests.get(
    url,
    params=params,
    headers={"User-Agent":"Mozilla/5.0"}
).json()

for r in data["query"]["search"][:10]:
    print(r["title"])

Brandon Sanderson bibliography
The Emperor's Soul
Robert Jordan bibliography
Dan Wells (author)
Brian McClellan
Charlie N. Holmberg
Dave Wolverton
Robison Wells
Chris Claremont
Amal El-Mohtar


In [ ]:
def get_bibliography_page(author):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": author + " bibliography",
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers={
            "User-Agent":"Mozilla/5.0"
        },
        timeout=30
    )

    data = response.json()

    results = data["query"]["search"]

    # Prefer bibliography pages
    for r in results:

        title = r["title"]

        if "bibliography" in title.lower():
            return title

    if len(results) > 0:
        return results[0]["title"]

    return None

In [ ]:
for author in [
    "Brandon Sanderson",
    "Robert Jordan",
    "Terry Pratchett",
    "Robin Hobb"
]:
    print(author, "->",
          get_bibliography_page(author))

Brandon Sanderson -> Brandon Sanderson bibliography
Robert Jordan -> Robert Jordan bibliography
Terry Pratchett -> Stephen Baxter bibliography
Robin Hobb -> Robin Hobb bibliography


In [ ]:
import time
import requests

def get_bibliography_page(author):

    time.sleep(1)

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": author + " bibliography",
        "format": "json"
    }

    try:

        response = requests.get(
            url,
            params=params,
            headers={
                "User-Agent":
                "BookShelfRecommenderBot/1.0"
            },
            timeout=30
        )

        if response.status_code != 200:
            print("Status:", response.status_code)
            return None

        try:
            data = response.json()
        except:
            print("Not JSON:", author)
            print(response.text[:200])
            return None

        results = data["query"]["search"]

        for r in results:

            if "bibliography" in r["title"].lower():
                return r["title"]

        if results:
            return results[0]["title"]

        return None

    except Exception as e:

        print(author, e)
        return None

In [ ]:
print(len(all_books))

pd.DataFrame(all_books).shape

397


(397, 2)

In [ ]:
import requests
import pandas as pd
import time
import re

from io import StringIO

In [ ]:
def get_bibliography_page(author):

    time.sleep(1)

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": author + " bibliography",
        "format": "json"
    }

    try:

        response = requests.get(
            url,
            params=params,
            headers={
                "User-Agent":
                "BookShelfRecommenderBot/1.0"
            },
            timeout=30
        )

        if response.status_code != 200:

            print(
                author,
                "status:",
                response.status_code
            )

            return None

        try:

            data = response.json()

        except:

            print(
                author,
                "JSON decode failed"
            )

            return None

        results = data["query"]["search"]

        author_words = set(
            author.lower().split()
        )

        # Prefer bibliography pages that
        # actually contain the author name

        for r in results:

            title = r["title"]

            title_lower = title.lower()

            if "bibliography" not in title_lower:
                continue

            matches = sum(
                word in title_lower
                for word in author_words
            )

            if matches > 0:
                return title

        # fallback

        if len(results):

            return results[0]["title"]

        return None

    except Exception as e:

        print(author, e)

        return None

In [ ]:
def get_wiki_books(page):

    url = (
        "https://en.wikipedia.org/wiki/"
        + page.replace(" ", "_")
    )

    response = requests.get(
        url,
        headers={
            "User-Agent":
            "Mozilla/5.0"
        },
        timeout=30
    )

    tables = pd.read_html(
        StringIO(response.text)
    )

    books = []

    for table in tables:

        cols = [
            str(c).lower()
            for c in table.columns
        ]

        if not any(
            "title" in c
            for c in cols
        ):
            continue

        title_col = None

        for col in table.columns:

            if (
                "title"
                in str(col).lower()
            ):

                title_col = col
                break

        if title_col is None:
            continue

        titles = (
            table[title_col]
            .dropna()
            .astype(str)
            .tolist()
        )

        books.extend(titles)

    return books

In [ ]:
catalog["text"] = (
    catalog["title"].fillna("") + " " +
    catalog["author"].fillna("") + " " +
    catalog["genres"].fillna("") + " " +
    catalog["description"].fillna("")
)

NameError: name 'catalog' is not defined